# Cubo OLAP con PySpark en Google Colab

Este notebook está preparado para ejecutarse en Colab y usa el modelo en estrella exportado desde SQL Server. Se clona el repositorio, se instala PySpark y se crean visualizaciones OLAP.

## 1. Preparación del entorno

Instala las dependencias necesarias y clona el repositorio para acceder a los CSV.

In [ ]:
!pip install --quiet pyspark pandas matplotlib seaborn
!git clone --depth 1 https://github.com/ftmrocca/CUBO-OLAP.git /content/CUBO-OLAP || true
import os
base_path = '/content/CUBO-OLAP'
assert os.path.exists(base_path), 'El repositorio no se clonó correctamente.'
print('base_path:', base_path)

## 2. Crear sesión Spark y cargar datos

Creamos la sesión de Spark, cargamos cada dimensión y la tabla de hechos desde los CSV.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when, round, sum as spark_sum

spark = SparkSession.builder
    .appName('CuboOLAPColab')
    .master('local[*]')
    .getOrCreate()

dim_candidate = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_candidate.csv')

dim_education = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .option('quote', 
)
    .csv(f'{base_path}/dim_education.csv')

dim_skills = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_skills.csv')

dim_time = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_time.csv')

fact_employability = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/fact_employability.csv')

print('Datos cargados con éxito')
print('dim_candidate rows:', dim_candidate.count())
print('fact_employability rows:', fact_employability.count())

## 3. Construcción del cubo OLAP

Unimos las dimensiones con la tabla de hechos para crear el cubo analítico.

In [ ]:
cube_df = fact_employability
    .join(dim_candidate, 'candidate_id', 'left')
    .join(dim_education, 'education_id', 'left')
    .join(dim_skills, 'skills_id', 'left')
    .join(dim_time, 'time_id', 'left')

cube_df.createOrReplaceTempView('cube')
cube_df.printSchema()

## 4. Visualizaciones OLAP

Estas celdas generan gráficos que muestran distintos cortes y agregaciones del cubo.

### 4.1 Distribución de candidatos por género y país

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid', palette='muted')

gender_country = cube_df.groupBy('gender', 'country_of_origin').agg(count('*').alias('count')).orderBy('gender', 'country_of_origin')
pdf = gender_country.toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pdf, x='country_of_origin', y='count', hue='gender', ax=ax)
ax.set_title('Distribución de candidatos por género y país de origen')
ax.set_xlabel('País de origen')
ax.set_ylabel('Número de candidatos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.2 Tasa de empleo por nivel educativo

In [ ]:
education_employment = cube_df.groupBy('education_level').agg(
    count('*').alias('total'),
    spark_sum(when(col('employment_status') == 'Employed', 1).otherwise(0)).alias('employed')
)
education_employment = education_employment.withColumn('employment_rate', round(col('employed') / col('total') * 100, 2))
pdf = education_employment.toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pdf, x='education_level', y='employment_rate', palette='Blues_d', ax=ax)
ax.set_title('Tasa de empleo por nivel educativo')
ax.set_xlabel('Nivel educativo')
ax.set_ylabel('Tasa de empleo (%)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

### 4.3 Salario promedio por sector y país

In [ ]:
salary_sector = cube_df.groupBy('job_sector', 'country_of_origin').agg(round(avg('salary'), 2).alias('avg_salary')).orderBy('job_sector', 'country_of_origin')
pdf = salary_sector.toPandas()
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=pdf, x='job_sector', y='avg_salary', hue='country_of_origin', ax=ax)
ax.set_title('Salario promedio por sector y país de origen')
ax.set_xlabel('Sector laboral')
ax.set_ylabel('Salario promedio')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 4.4 Experiencia de prácticas vs años desde graduación

In [ ]:
experience_grad = cube_df.groupBy('years_since_graduation', 'internship_experience').agg(count('*').alias('count')).orderBy('years_since_graduation')
pdf = experience_grad.toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pdf, x='years_since_graduation', y='count', hue='internship_experience', ax=ax)
ax.set_title('Candidatos por años desde graduación y experiencia de prácticas')
ax.set_xlabel('Años desde graduación')
ax.set_ylabel('Cantidad de candidatos')
plt.tight_layout()
plt.show()

### 4.5 Salario promedio por ranking universitario

In [ ]:
salary_ranking = cube_df.groupBy('university_ranking').agg(round(avg('salary'), 2).alias('avg_salary')).orderBy('university_ranking')
pdf = salary_ranking.toPandas()
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=pdf, x='university_ranking', y='avg_salary', palette='viridis', ax=ax)
ax.set_title('Salario promedio por ranking universitario')
ax.set_xlabel('Ranking de la universidad')
ax.set_ylabel('Salario promedio')
plt.tight_layout()
plt.show()

## 5. Conclusión

Este notebook muestra cómo crear un cubo OLAP con PySpark a partir de un modelo en estrella y cómo visualizar los resultados en Colab.